In [2]:
import random
random.seed(42)


AA_ONE = {
    "AUG": "M",
    "UUU": "F", "UUC": "F",
    "UUA": "L", "UUG": "L",
    "ACC": "T", "ACU": "T", "ACA": "T", "ACG": "T",
    "UGG": "W",
    "GGG": "G", "GGA":"G", "GGC": "G", "GGU":"G",
    "CCU": "P", "CCC" : "P", "CCA" : "P", "CCG" : "P",
    "UAA": "*", "UAG": "*", "UGA": "*"
}


def gc_content(seq):
    return (seq.count("G") + seq.count("C")) / len(seq)





def find_orfs(rna, codon_map):
    assert set(rna).issubset({"A", "U", "G", "C"}), "Invalid RNA"

    orfs = []

    for frame in range(3):
        i = frame
        while i <= len(rna) - 3:
            codon = rna[i:i+3]
            frame_penalty = 0 if frame == 0 else 1

            if codon == "AUG":
                protein = ""
                start = i
                j = i

                while j <= len(rna) - 3:
                    aa = codon_map.get(rna[j:j+3])

                    if aa == "*":
                        nt_length = (j + 3) - start
                        orf_rna = rna[start:j+3]
                        score = (len(protein) * 2) + (gc_content(orf_rna) * 10) - (frame_penalty * 5)
                        orfs.append({
                            "frame": frame + 1,
                            "start": start,
                            "stop": j + 2,
                            "protein": protein,
                            "nt_length": nt_length,
                            "gc_content": gc_content(orf_rna),
                            "score": score
                        })
                        break

                    if aa:
                        protein += aa

                    j += 3

                i = j
            else:
                i += 3

    return orfs







def point_mutation(rna):
    pos = random.randint(0, len(rna) - 1)
    nts = ["A", "U", "G", "C"]
    nts.remove(rna[pos])
    mutated = rna[:pos] + random.choice(nts) + rna[pos+1]
    return mutated, "point", pos



def deletion_mutation(rna):
    pos = random.randint(0, len(rna) - 1)
    mutated = rna[:pos] + rna[pos+1:]
    return mutated, "deletion", pos



def insertion_mutation(rna):
    pos = random.randint(0, len(rna))
    nt = random.choice(["A", "U", "G", "C"])
    mutated = rna[:pos] + nt + rna[pos:]
    return mutated, "insertion", pos




def compare_orfs(original, mutated):
    lost_orfs = max(0, len(original) - len(mutated))
    
    if original and mutated:
        longest_orig = max(original, key=lambda x: len(x["protein"]))["protein"]
        longest_mut = max(mutated, key=lambda x: len(x["protein"]))["protein"]
        protein_loss = max(0, len(longest_orig) - len(longest_mut))
        
    else:
        protein_loss = 0
        
    return lost_orfs, protein_loss    



def severity_score(lost_orfs, protein_loss, frameshift):
    return (lost_orfs * 5) + (protein_loss * 2) + (frameshift * 10)




rna = "AUGGGGCCCUAA"
original_orfs = find_orfs(rna, AA_ONE)

pt_rna, t1, _ = point_mutation(rna)
del_rna, t2, _ = deletion_mutation(rna)
ins_rna, t3, _ = insertion_mutation(rna)

pt_orfs = find_orfs(pt_rna, AA_ONE)
del_orfs = find_orfs(del_rna, AA_ONE)
ins_orfs = find_orfs(ins_rna, AA_ONE)

pt_lost, pt_loss = compare_orfs(original_orfs, pt_orfs)
del_lost, del_loss = compare_orfs(original_orfs, del_orfs)
ins_lost, ins_loss = compare_orfs(original_orfs, ins_orfs)

pt_score = severity_score(pt_lost, pt_loss, 0)
del_score = severity_score(del_lost, del_loss, 1)
ins_score = severity_score(ins_lost, ins_loss, 1)

assert del_score > pt_score
assert ins_score > pt_score
